In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.interpolate import UnivariateSpline
from scipy.integrate import trapezoid, cumulative_trapezoid
from datetime import datetime

# --- CONFIGURATION ---
TICKER = "^STOXX50E" # Correction du ticker pour Yahoo Finance (Indice Euro Stoxx 50)
CAPITAL_VIRTUEL = 10000
HORIZON_DAYS = 5
R, Q = 0.043, 0.015

def plot_probability_density(strikes, pdf, S0, mode, stop_loss):
    plt.figure(figsize=(10, 6))
    plt.plot(strikes, pdf, label='Distribution de Probabilité (PDF)', color='blue', lw=2)
    plt.axvline(S0, color='black', linestyle='--', label=f'Prix Actuel ({S0:.2f})')
    plt.axvline(mode, color='green', linestyle='-', label=f'Objectif / Mode ({mode:.2f})')
    plt.axvline(stop_loss, color='red', linestyle='-', label=f'Stop Loss ({stop_loss:.2f})')
    
    # Remplissage de la zone de profit probable
    plt.fill_between(strikes, pdf, where=(strikes >= S0), color='green', alpha=0.2, label='Zone de Profit')
    
    plt.title(f"Analyse Quantitative des Options - {TICKER}")
    plt.xlabel("Prix à l'échéance")
    plt.ylabel("Probabilité")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

def quant_decision_engine(ticker_symbol):
    asset = yf.Ticker(ticker_symbol)
    hist = asset.history(period="1d")
    if hist.empty: return None
    S0 = hist['Close'].iloc[-1]
    
    expirations = asset.options
    if not expirations: return None
    
    target_date = expirations[np.argmin([abs((pd.to_datetime(d) - datetime.now()).days - HORIZON_DAYS) for d in expirations])]
    T = max((pd.to_datetime(target_date) - datetime.now()).days, 1) / 365
    
    opt = asset.option_chain(target_date)
    calls = opt.calls[(opt.calls['bid'] > 0) & (opt.calls['ask'] > 0)].copy()
    calls = calls[(calls['strike'] > S0 * 0.85) & (calls['strike'] < S0 * 1.15)]
    
    if len(calls) < 8: return None

    # Spline pour lisser le sourire de volatilité
    spline = UnivariateSpline(calls['strike'], calls['impliedVolatility'], k=3, s=0.01)
    strikes = np.linspace(calls['strike'].min(), calls['strike'].max(), 1000)
    dk = strikes[1] - strikes[0]
    
    def bs_price(K, sig):
        d1 = (np.log(S0/K) + (R - Q + 0.5*sig**2)*T) / (sig*np.sqrt(T))
        d2 = d1 - sig*np.sqrt(T)
        return S0*np.exp(-Q*T)*norm.cdf(d1) - K*np.exp(-R*T)*norm.cdf(d2)
    
    prices = np.array([bs_price(K, spline(K)) for K in strikes])
    
    # Calcul de la PDF (Dérivée seconde du prix par rapport au strike)
    pdf = np.maximum(np.gradient(np.gradient(prices, dk), dk) * np.exp(R * T), 0)
    pdf /= trapezoid(pdf, strikes) # Normalisation
    cdf = cumulative_trapezoid(pdf, strikes, initial=0)

    mode = strikes[np.argmax(pdf)]
    pop = (1 - cdf[np.argmin(np.abs(strikes - S0))]) * 100
    stop_loss = strikes[np.argmin(np.abs(cdf - 0.05))]
    
    # Affichage du graphique
    plot_probability_density(strikes, pdf, S0, mode, stop_loss)
    
    b = (mode/S0 - 1) / abs(stop_loss/S0 - 1) if stop_loss != S0 else 0
    kelly_f = max(0, (pop/100 * (b + 1) - 1) / b) if b > 0 else 0
    
    return {'S0': S0, 'TP': mode, 'SL': stop_loss, 'PoP': pop, 'Kelly': kelly_f}



# Lancer l'analyse
decision = quant_decision_engine(TICKER)
if decision:
    print(f"\n--- RÉSULTATS ---")
    print(f"Prix actuel: {decision['S0']:.2f}")
    print(f"Objectif (Mode): {decision['TP']:.2f}")
    print(f"Probabilité de profit (PoP): {decision['PoP']:.1f}%")
    print(f"Allocation Kelly: {decision['Kelly']*100:.1f}%")

import time

# ... (gardez toutes les fonctions précédentes : quant_decision_engine, etc.)

def run_scanner():
    print(f"🚀 Lancement du scanner sur {TICKER}...")
    
    while True:
        try:
            decision = quant_decision_engine(TICKER)
            
            if decision and decision['Kelly'] > 0:
                print(f"\n🎯 OPPORTUNITÉ DÉTECTÉE !")
                print(f"Probabilité de Profit (PoP): {decision['PoP']:.1f}%")
                print(f"Allocation Kelly: {decision['Kelly']*100:.1f}%")
                
                # Une fois l'opportunité trouvée, on passe au trading réel (ou paper)
                start_paper_trading_with_log(decision) 
                break # On sort de la boucle de scan car on est en position
                
            else:
                current_time = datetime.now().strftime('%H:%M:%S')
                print(f"😴 {current_time} : Pas d'avantage statistique. Prochain scan dans 15 min.", end='\r')
                time.sleep(10) # Attendre 15 minutes (900 secondes)
                
        except Exception as e:
            print(f"⚠️ Erreur lors du scan : {e}")
            time.sleep(60) # Attendre 1 min avant de réessayer en cas d'erreur réseau

run_scanner()

🚀 Lancement du scanner sur ^STOXX50E...


KeyboardInterrupt: 